In [27]:
from netrc import netrc

import  torch
from  torch import  nn
from  d2l import  torch as d2l


In [28]:
# 使用 Sequential 容器快速构建模型，包含 Flatten、全连接层、激活函数和输出层
net = nn.Sequential(
    nn.Flatten(),              # 将输入的 28x28 图像展平成一维向量（784 维）
    nn.Linear(784, 256),       # 第一层：线性变换，输入784维 → 输出256维
    nn.ReLU(),                 # 激活函数：引入非线性
    nn.Linear(256, 10)         # 输出层：将256维映射到10个类别（对应Fashion-MNIST的10类）
)
def init_weights(m):
    if type(m)==nn.Linear:
        nn.init.normal_(m.weight,std=0.01)
net.apply(init_weights)

Sequential(
  (0): Flatten(start_dim=1, end_dim=-1)
  (1): Linear(in_features=784, out_features=256, bias=True)
  (2): ReLU()
  (3): Linear(in_features=256, out_features=10, bias=True)
)

In [29]:
batch_size,lr,num_epochs=256,0.1,10
loss=nn.CrossEntropyLoss(reduction='none')
trainer=torch.optim.SGD(net.parameters(),lr=lr)

train_iter,test_iter=d2l.load_data_fashion_mnist(batch_size)


In [30]:
def train_loop():
    # 循环训练多个 epoch
    for epoch in range(num_epochs):
        # 初始化累积变量，用于记录总损失、正确预测数、样本数
        train_loss_sum, train_acc_sum, n = 0.0, 0.0, 0
        # 遍历训练数据集中的每个 batch
        for X, y in train_iter:
            # 前向传播：计算预测输出
            y_hat = net(X)
            # 计算交叉熵损失
            l = loss(y_hat, y).mean()
            # 清空上一次迭代的梯度
            trainer.zero_grad()
            # 反向传播：计算参数梯度
            l.backward()
            # 用优化器更新参数
            trainer.step()
            # 累积当前 batch 的损失值
            train_loss_sum += l.item()
            # 计算预测正确的样本数
            train_acc_sum += (y_hat.argmax(dim=1) == y).sum().item()
            # 累积总样本数
            n += y.shape[0]

        # 计算测试集上的准确率
        test_acc_sum, test_n = 0.0, 0
        with torch.no_grad():  # 关闭梯度计算，加快测试速度
            for X, y in test_iter:
                # 计算预测结果
                y_hat = net(X)
                # 累积正确预测的样本数
                test_acc_sum += (y_hat.argmax(dim=1) == y).sum().item()
                # 累积总样本数
                test_n += y.shape[0]

        # 打印每个 epoch 的统计结果
        print(f'epoch {epoch + 1}, loss {100 * train_loss_sum / n:.2f}%, '
              f'train acc {100 * train_acc_sum / n:.2f}%, '
              f'test acc {100 * test_acc_sum / test_n:.2f}%')



train_loop()

epoch 1, loss 0.41%, train acc 63.53%, test acc 72.55%
epoch 2, loss 0.24%, train acc 78.86%, test acc 78.21%
epoch 3, loss 0.20%, train acc 81.81%, test acc 73.92%
epoch 4, loss 0.19%, train acc 83.12%, test acc 81.56%
epoch 5, loss 0.18%, train acc 84.16%, test acc 82.39%
epoch 6, loss 0.17%, train acc 84.73%, test acc 79.49%
epoch 7, loss 0.16%, train acc 85.20%, test acc 82.37%
epoch 8, loss 0.16%, train acc 85.74%, test acc 83.98%
epoch 9, loss 0.15%, train acc 86.18%, test acc 84.88%
epoch 10, loss 0.15%, train acc 86.37%, test acc 84.09%
